In [0]:
def load_sql_queries():
    """
    Loads SQL query files from workspace path.
    
    Returns:
        DataFrame with columns: table_name, query
        None if no valid queries found
    """
    try:
        # List files using dbutils
        try:
            files = dbutils.fs.ls(QUERY_PATH)
        except Exception as e:
            logger.error(f"Query path does not exist or is inaccessible: {QUERY_PATH}")
            raise
            
        sql_files = [file.path for file in files if file.path.endswith('.sql')]
        
        if not sql_files:
            logger.warning(f"No SQL files found in {QUERY_PATH}")
            return None
        
        logger.info(f"Found {len(sql_files)} SQL files in {QUERY_PATH}")
        
        # Read SQL files into a list of tuples (table_name, query)
        queries = []
        for file_path in sql_files:
            try:
                # Extract table name from filename
                table_name = file_path.split('/')[-1].replace('.sql', '')
                local_path = file_path.replace('dbfs:', '')
                
                # Read the file using Python's built-in open
                with open(local_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                if content.strip():
                    queries.append((table_name, content.strip()))
                    logger.info(f"Loaded query: {table_name} ({len(content)} chars)")
                else:
                    logger.warning(f"Skipped empty file: {table_name}")
                    
            except Exception as e:
                logger.error(f"Failed to read {file_path}: {e}")
                # Continue processing other files
                continue
        
        if not queries:
            logger.warning(f"All SQL files in {QUERY_PATH} are empty or failed to load")
            return None
        
        # Create DataFrame from the list
        df = spark.createDataFrame(queries, ["table_name", "query"])
        logger.info(f"Successfully loaded {df.count()} SQL queries")
        return df
        
    except Exception as e:
        logger.error(f"Failed to load SQL files: {e}")
        logger.error(traceback.format_exc())
        raise